# 手游留存策略 A/B 实验分析

## 项目简介
基于 Cookie Cats 手游真实实验数据（9万用户），评估关卡门位置调整对用户留存率的影响，
为版本迭代提供统计决策依据。

## 实验背景
Cookie Cats 是一款消除类手游。玩家在推进关卡时会遇到"关卡门"，需等待一段时间或付费才能继续。  
本次实验对比两种策略：
- **gate_30（对照组）**：关卡门设在第30关（现有方案）
- **gate_40（实验组）**：关卡门推迟至第40关（新方案）

**实验问题：** 把门往后移，玩家会玩得更久吗？

## 数据来源
Kaggle 公开数据集：[Mobile Games AB Testing Cookie Cats](https://www.kaggle.com/datasets/mursideyarkin/mobile-games-ab-testing-cookie-cats)

| 字段 | 含义 |
|------|------|
| userid | 用户ID |
| version | 所属实验组（gate_30 / gate_40）|
| sum_gamerounds | 安装后14天内总游戏轮数 |
| retention_1 | 次日是否留存（True/False）|
| retention_7 | 7日后是否留存（True/False）|

## 分析目录
1. 数据概览
2. 实验有效性验证（SRM检验）
3. 基础指标对比
4. 显著性检验（卡方检验）
5. 实验结论与建议


## Part 1：数据概览

In [ ]:
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'SimHei'
matplotlib.rcParams['axes.unicode_minus'] = False

# 数据文件路径请根据本地实际路径修改
df_ab = pd.read_csv('cookie_cats.csv')

print(f'数据量：{df_ab.shape[0]} 行，{df_ab.shape[1]} 列')
print()
print(df_ab.info())
print()
print(df_ab.head())


## Part 2：实验有效性验证（SRM 检验）

In [ ]:
# SRM（Sample Ratio Mismatch）检验：验证两组分配是否符合预期比例（1:1）
# 若两组人数严重不均，说明随机分组存在问题，实验结果不可信

version_counts = df_ab['version'].value_counts()
print('=== 两组样本量 ===')
print(version_counts)
print()

# 用卡方检验验证是否偏离1:1分配
total = version_counts.sum()
expected = [total/2, total/2]
observed = [version_counts['gate_30'], version_counts['gate_40']]
chi2, p = stats.chisquare(observed, expected)

print(f'SRM卡方检验：chi2={chi2:.4f}, p={p:.4f}')
if p > 0.05:
    print('结论：两组分配无显著偏差（p>0.05），随机分组质量良好，实验有效')
else:
    print('警告：两组分配存在显著偏差，实验可能存在SRM问题，需排查原因')


## Part 3：基础指标对比

In [ ]:
summary = df_ab.groupby('version').agg(
    用户数=('userid','count'),
    次日留存率=('retention_1','mean'),
    七日留存率=('retention_7','mean'),
    平均游戏轮数=('sum_gamerounds','mean')
).round(4)
summary['次日留存率%'] = (summary['次日留存率'] * 100).round(2)
summary['七日留存率%'] = (summary['七日留存率'] * 100).round(2)
print(summary[['用户数','次日留存率%','七日留存率%','平均游戏轮数']])


## Part 4：显著性检验（卡方检验）

In [ ]:
# 留存率是二分类指标（True/False），使用卡方检验比较两组比例差异
# 若为连续数值指标（如游戏轮数），则应使用t检验

gate30 = df_ab[df_ab['version'] == 'gate_30']
gate40 = df_ab[df_ab['version'] == 'gate_40']

def ab_test(group_a, group_b, metric, name):
    a_total, b_total = len(group_a), len(group_b)
    a_convert = group_a[metric].sum()
    b_convert = group_b[metric].sum()
    a_rate = a_convert / a_total
    b_rate = b_convert / b_total

    contingency = [[a_convert, a_total-a_convert],
                   [b_convert, b_total-b_convert]]
    chi2, p_value, _, _ = stats.chi2_contingency(contingency)
    lift = (a_rate - b_rate) / b_rate * 100

    print(f'=== {name} ===')
    print(f'  gate_30: {a_convert}/{a_total} = {a_rate*100:.2f}%')
    print(f'  gate_40: {b_convert}/{b_total} = {b_rate*100:.2f}%')
    print(f'  相对提升: {lift:+.2f}%')
    print(f'  卡方值: {chi2:.4f},  p值: {p_value:.4f}')
    sig = p_value < 0.05
    print(f'  结论：{"差异显著（p<0.05）✓" if sig else "差异不显著（p≥0.05）"}')
    print()
    return p_value

p1 = ab_test(gate30, gate40, 'retention_1', '次日留存率 Day-1')
p7 = ab_test(gate30, gate40, 'retention_7', '七日留存率 Day-7')


In [ ]:
# 可视化
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
versions = ['gate_30\n（对照组）', 'gate_40\n（实验组）']
colors = ['#2ecc71', '#e74c3c']

d1 = [44.82, 44.23]
bars1 = axes[0].bar(versions, d1, color=colors, width=0.4)
axes[0].set_ylim(43, 46)
axes[0].set_title(f'次日留存率对比\n(p=0.0755，差异不显著)', fontsize=11)
axes[0].set_ylabel('留存率 %')
for bar, val in zip(bars1, d1):
    axes[0].text(bar.get_x()+bar.get_width()/2, val+0.05,
                 f'{val}%', ha='center', fontsize=12, fontweight='bold')

d7 = [19.02, 18.20]
bars2 = axes[1].bar(versions, d7, color=colors, width=0.4)
axes[1].set_ylim(17, 20.5)
axes[1].set_title(f'七日留存率对比\n(p=0.0016，差异显著 ✓)', fontsize=11)
axes[1].set_ylabel('留存率 %')
for bar, val in zip(bars2, d7):
    axes[1].text(bar.get_x()+bar.get_width()/2, val+0.05,
                 f'{val}%', ha='center', fontsize=12, fontweight='bold')

axes[1].annotate('gate_30显著更优\n提升+4.51%',
                 xy=(0, 19.02), xytext=(0.6, 19.8),
                 arrowprops=dict(arrowstyle='->', color='black'),
                 fontsize=10, color='#2ecc71', fontweight='bold')

plt.suptitle('Cookie Cats A/B 实验结果', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## Part 5：实验结论与建议

In [ ]:
print('='*50)
print('实验结论')
print('='*50)
print()
print('1. 次日留存（Day-1）：')
print('   gate_30=44.82% vs gate_40=44.23%，p=0.0755')
print('   差异不显著——短期内两组玩家体验无明显差别')
print()
print('2. 七日留存（Day-7）：')
print('   gate_30=19.02% vs gate_40=18.20%，p=0.0016')
print('   差异显著——长期来看关卡门前置（第30关）更能留住玩家')
print()
print('3. 建议：保留现有方案（gate_30）')
print()
print('4. 方法论启示：')
print('   单纯依赖次日留存可能导致错误决策（p=0.076接近0.05，易误判）')
print('   留存类指标需结合短期+长期综合评估，避免以单一指标定结论')
